In [1]:
import h5py
import quimb as qu
import quimb.tensor as qtn
import numpy as np
import copy

import torch
from torch import optim
import tqdm
import cotengra as ctg

opti = ctg.ReusableHyperOptimizer(
    progbar=True,
    methods=['greedy'],
    reconf_opts={},
    max_repeats=32, 
    optlib='random',
    # directory=  # set this for persistent cache
)

### Read data

In [3]:
def read_data(data_name):
    with h5py.File("save_results/" + data_name + ".h5", "r") as f:
        keys = sorted(f.keys(), key=lambda x: int(x.split("_")[1]))
        # print(keys)
        data = [np.transpose(f[key][:], (3,2,1,0)) for key in keys]

    return data

In [4]:
def array_to_lpdo(M1, tags):
    # convert input list of arrays to LPDO

    L = len(M1)

    inds = ('s0','e0','l0')
    first_tensor = M1[0][0,:,:,:]
    last_tensor = M1[-1][:,:,:,0]
    lpdo_1 = qtn.Tensor(data=first_tensor, inds=inds, tags=tags)

    for i in range(1, L):
        if i == L-1:
            inds = (f'l{i-1}', f's{i}', f'e{i}')
            current_tensor = qtn.Tensor(data=last_tensor, inds=inds, tags=tags)
        else:
            # bond, system, environment, bond
            inds = (f'l{i-1}', f's{i}', f'e{i}', f'l{i}')
            current_tensor = qtn.Tensor(data=M1[i], inds=inds, tags=tags)

        lpdo_1 = lpdo_1 & current_tensor

    return lpdo_1

In [5]:
def add_ancilla(lpdo,label):
    
    L = len(lpdo.tensors)
    lpdo_acl = lpdo.copy()

    for i in range(L):
        prod = qtn.Tensor(np.array([1,0]),inds = (label+f'{i}',),tags = 'A')
        # prod.apply_to_arrays(lambda x: torch.tensor(x, dtype=torch.complex128))
        
        # like direct product (outer product)
        lpdo_acl = lpdo_acl & prod

    return lpdo_acl

## Use Torch

In [6]:
def brickwall_unitary(psi, n_apply, list_u3, depth, n_Qbit, val_iden = 0,rand = False,start_layer=0,is_acl=0):

    if n_Qbit==0: depth=1
    if n_Qbit==1: depth=1

    for r in range(depth):

        if (r+start_layer)%2==0:
            if is_acl==0:
                for i in range(0, n_Qbit, 2):
                    # print("U_e", i, i + 1, n_apply)

                    if rand == True:
                        G = qu.rand_uni(4, dtype=complex)
                        #G = qu.fsimg(1,1,1,1,1, dtype=complex)
                    else:
                        G = qu.identity(4,dtype=complex)+qu.rand_uni(4, dtype=complex)*val_iden
                
                    psi.gate_(G, (i, i + 1), tags={'U',f'G{n_apply}', f'L{i}D{r}'})
                    list_u3.append(f'G{n_apply}')
                    n_apply+=1

            elif is_acl==1:
                for i in range(0, n_Qbit, 4):

                    if rand == True:
                        G = qu.rand_uni(16, dtype=complex)
                    else:
                        G = qu.identity(16,dtype=complex)+qu.rand_uni(16, dtype=complex)*val_iden
                
                    psi.gate_(G, (i, i + 1,i+2,i+3), tags={'U',f'G{n_apply}', f'L{i}D{r}'})
                    list_u3.append(f'G{n_apply}')
                    n_apply+=1

        else:
            if is_acl==0:
                for i in range(0, n_Qbit-2, 2):
                    # print("U_o", i+1, i + 2, n_apply)
            
                    if rand == True:
                        G = qu.rand_uni(4, dtype=complex)
                        #G = qu.fsimg(1,1,1,1,1, dtype=complex)
                    else:
                        G = qu.identity(4,dtype=complex)+qu.rand_uni(4, dtype=complex)*val_iden

                    psi.gate_(G, (i+1, i + 2), tags={'U',f'G{n_apply}', f'L{i}D{r}'})
                    list_u3.append(f'G{n_apply}')
                    n_apply+=1

            elif is_acl == 1:
                for i in range(2, n_Qbit-2, 4):
                    # print("U_o", i, i + 1,i+2,i+3, n_apply)
            
                    if rand == True:
                        G = qu.rand_uni(16, dtype=complex)
                        #G = qu.fsimg(1,1,1,1,1, dtype=complex)
                    else:
                        G = qu.identity(16,dtype=complex)+qu.rand_uni(16, dtype=complex)*val_iden

                    psi.gate_(G, (i, i + 1,i+2,i+3), tags={'U',f'G{n_apply}', f'L{i}D{r}'})
                    list_u3.append(f'G{n_apply}')
                    n_apply+=1

    return n_apply, list_u3


def staircase_unitary(psi, n_apply, list_u3, depth, n_Qbit, val_iden = 0,rand = False,start_layer=0, is_acl=0):

    if n_Qbit==0: depth=1
    if n_Qbit==1: depth=1

    for r in range(depth):

        if (r+start_layer)%2==0:
            if is_acl == 0:
                for i in range(0, n_Qbit-1, 1):
                    # print("U_e", i, i + 1, n_apply)

                    if rand == True:
                        G = qu.rand_uni(4, dtype=complex)
                        #G = qu.fsimg(1,1,1,1,1, dtype=complex)
                    else:
                        G = qu.identity(4,dtype=complex)+qu.rand_uni(4, dtype=complex)*val_iden
                
                    psi.gate_(G, (i, i + 1), tags={'U',f'G{n_apply}',f'L{i}D{r}'})
                    list_u3.append(f'G{n_apply}')
                    n_apply+=1

            elif is_acl == 1:
                # acts on four sites, two system, two ancilla
                for i in range(0, n_Qbit-3, 2):
                    # print("U_e", i, i + 1, n_apply)

                    if rand == True:
                        G = qu.rand_uni(16, dtype=complex)
            
                    else:
                        G = qu.identity(16,dtype=complex)+qu.rand_uni(16, dtype=complex)*val_iden
                
                    psi.gate_(G, (i, i + 1, i+2, i+3), tags={'U',f'G{n_apply}',f'L{i}D{r}'})
                    list_u3.append(f'G{n_apply}')
                    n_apply+=1

        else:
            if is_acl == 0:
                for i in range(n_Qbit-1, 0, -1):
                    # print("U_o", i-1, i, n_apply)
            
                    if rand == True:
                        G = qu.rand_uni(4, dtype=complex)
                        #G = qu.fsimg(1,1,1,1,1, dtype=complex)
                    else:
                        G = qu.identity(4,dtype=complex)+qu.rand_uni(4, dtype=complex)*val_iden

                    psi.gate_(G, (i-1, i), tags={'U',f'G{n_apply}',f'L{i}D{r}'})
                    list_u3.append(f'G{n_apply}')
                    n_apply+=1

            elif is_acl == 1:
                for i in range(n_Qbit-1, 2, -2):
                    # print("U_o", i-1, i, n_apply)
            
                    if rand == True:
                        G = qu.rand_uni(16, dtype=complex)
                        #G = qu.fsimg(1,1,1,1,1, dtype=complex)
                    else:
                        G = qu.identity(16,dtype=complex)+qu.rand_uni(16, dtype=complex)*val_iden

                    psi.gate_(G, (i-3, i-2, i-1, i), tags={'U',f'G{n_apply}',f'L{i}D{r}'})
                    list_u3.append(f'G{n_apply}')
                    n_apply+=1

    return n_apply, list_u3


def qmps_f(L=16, in_depth=2, val_iden = 0, rand = True,start_layer = 0, framework='brickwall', is_acl=0):

    list_u3=[]
    n_apply=0
    psi = qtn.MPS_computational_state('0' * (L))
    for i in range(L):
        t = psi[i]
        indx = 'k'+str(i)
        t.modify(left_inds=[indx])

    for t in range(L):
        psi[t].modify(tags=[f"I{t}", "MPS"])

    if framework == 'brickwall':
        n_apply, list_u3=brickwall_unitary(psi, n_apply, list_u3, in_depth, L, val_iden = val_iden, rand =rand,start_layer=start_layer,is_acl=is_acl)
    elif framework == 'staircase':
        n_apply, list_u3=staircase_unitary(psi, n_apply, list_u3, in_depth, L, val_iden = val_iden, rand =rand,start_layer=start_layer,is_acl=is_acl)
    elif framework == 'mixed':
        n_apply, list_u3=brickwall_unitary(psi, n_apply, list_u3, in_depth, L, val_iden = val_iden, rand =rand,start_layer=start_layer,is_acl=is_acl)
        n_apply, list_u3=staircase_unitary(psi, n_apply, list_u3, in_depth, L, val_iden = val_iden, rand =rand,start_layer=start_layer,is_acl=is_acl)


    return psi.astype_('complex128')#, list_u3


def extract_unitary_circuit(psi_pqc, num_qubits):
    # only system qubits

    pqc = psi_pqc.tensors[num_qubits]
    for i in range (num_qubits+1,len(psi_pqc.tensors)):
        pqc = pqc&psi_pqc.tensors[i] #extrating the circuit part

    for i in range (num_qubits):
        pqc = pqc.reindex({f'k{i}':f'e{i}'})
        pqc = pqc.reindex({psi_pqc.tensors[i].inds[-1]:f'ep{i}'})

    return pqc


def extract_unitary_circuit_acl(psi_pqc, num_qubits):
    # for ancilla. num_qubits = 2*n

    pqc = psi_pqc.tensors[num_qubits]
    for i in range (num_qubits+1,len(psi_pqc.tensors)):
        pqc = pqc&psi_pqc.tensors[i] #extrating the circuit part

    for i in range (num_qubits):
        if (i%2):
            pqc = pqc.reindex({f'k{i}':f'e{i//2}'})
            pqc = pqc.reindex({psi_pqc.tensors[i].inds[-1]:f'ep{i//2}'})
        else:
            pqc = pqc.reindex({f'k{i}':f'a{i//2}'})
            pqc = pqc.reindex({psi_pqc.tensors[i].inds[-1]:f'ap{i//2}'})

    return pqc

In [7]:
def get_pqc_torch(n, depth, framework, is_acl, rand=True, val_iden=0.0):

    if is_acl == 0:
        num_qubits = n # physical 
        psi_pqc = qmps_f(num_qubits, in_depth= depth, val_iden = val_iden,rand = rand, framework=framework)
        pqc = extract_unitary_circuit(psi_pqc, num_qubits)
        # full_contraction(pqc, lpdo_1, lpdo_2, is_show=0)


    elif is_acl == 1:
        num_qubits = 2 * n # physical + ancilla 
        psi_pqc = qmps_f(num_qubits, in_depth= depth, val_iden = val_iden, rand = rand, framework=framework,is_acl=1)
        pqc = extract_unitary_circuit_acl(psi_pqc, num_qubits)
        # full_contraction(pqc, lpdo_1_acl, lpdo_2_acl, is_show=0)

    return pqc


def get_lpdo_torch(M1, M2, is_acl):

    n = len(M1)
    lpdo_1 = array_to_lpdo(M1, ('M1',))

    lpdo_2 = array_to_lpdo(M2, ('M2',))
    lpdo_2 = lpdo_2.H
    for i in range (n):
        lpdo_2 = lpdo_2.reindex({f'e{i}':f'ep{i}'})

    lpdo_1_acl = add_ancilla(lpdo_1,"a")
    lpdo_2_acl = add_ancilla(lpdo_2,"ap")

    if is_acl == 0:
        lpdo_1_torch = lpdo_1.copy()
        lpdo_2_torch = lpdo_2.copy()
    elif is_acl == 1:
        lpdo_1_torch = lpdo_1_acl.copy()
        lpdo_2_torch = lpdo_2_acl.copy()

    return lpdo_1_torch, lpdo_2_torch

def full_contraction(pqc, lpdo_1, lpdo_2, is_show=0):
    if is_show == 1:
        (lpdo_1 & lpdo_2 & pqc).draw(['U','M2','M1'])

    output = abs((lpdo_1 & lpdo_2 & pqc).contract(optimize=opti))
    
    # return torch.abs(output).real 
    return -output

In [8]:
class TNModel(torch.nn.Module):
    # this class is inheritance of torch.nn.Module

    def __init__(self, pqc, lpdo_1, lpdo_2):
        super().__init__()

        # extract the raw arrays and a skeleton of the TN
        params, self.skeleton = qtn.pack(pqc)
        # n.b. you might want to do extra processing here to e.g. store each
        # parameter as a reshaped matrix (from left_inds -> right_inds), for
        # some optimizers, and for some torch parametrizations

        self.torch_params = torch.nn.ParameterDict({
            # "str(i)" is key conversion: torch requires strings as keys
            str(i): torch.nn.Parameter(initial)
            for i, initial in params.items()
        })
        self._loss_fn = lambda x: full_contraction(x, lpdo_1, lpdo_2)
        
    def forward(self):
        # convert back to original int key format
        params = {int(i): p for i, p in self.torch_params.items()}
        # reconstruct the TN with the new parameters
        pqc = qtn.unpack(params, self.skeleton)
        
        # isometrize and then return the energy
        return self._loss_fn(pqc.isometrize(method='exp'))
    
    def forward_debug(self):
        # convert back to original int key format
        params = {int(i): p for i, p in self.torch_params.items()}
        # reconstruct the TN with the new parameters
        pqc = qtn.unpack(params, self.skeleton)
        
        # isometrize and then return the energy
        return self._loss_fn(pqc)
    
    def get_unitary_circuit(self):
        """Extract the optimized unitary quantum circuit."""
        # Convert parameters back to int keys
        params = {int(i): p for i, p in self.torch_params.items()}
        # Reconstruct the circuit
        pqc = qtn.unpack(params, self.skeleton)
        # Apply isometrization to get unitary gates
        return pqc.isometrize(method='exp')
    

### Optimization

In [22]:
M1 = read_data("M1_a2_N6")
M2 = read_data("M2_a2_N6")
n = len(M1)
is_acl = 1
depth = 2  # this should be twice the depth in the note
framework = 'mixed'
rand = True
val_iden = 0.0

pqc_init = get_pqc_torch(n, depth, framework, is_acl, rand=rand, val_iden=val_iden)
lpdo_1_torch, lpdo_2_torch = get_lpdo_torch(M1, M2, is_acl)

pqc_torch = pqc_init.copy()
pqc_init.apply_to_arrays(lambda x: torch.tensor(x, dtype=torch.complex128))
pqc_torch.apply_to_arrays(lambda x: torch.tensor(x, dtype=torch.complex128))
lpdo_1_torch.apply_to_arrays(lambda x: torch.tensor(x, dtype=torch.complex128))
lpdo_2_torch.apply_to_arrays(lambda x: torch.tensor(x, dtype=torch.complex128))

In [18]:
model = TNModel(pqc_torch, lpdo_1_torch, lpdo_2_torch)
# print(full_contraction(lpdo_1_torch, lpdo_2_torch, pqc_init))
# print(full_contraction(lpdo_1_torch, lpdo_2_torch, pqc_torch))
# verify_unitarity(pqc_init)

print("Model created! Initial loss:", model.forward().item())

# print(full_contraction(lpdo_1_torch, lpdo_2_torch, pqc_init))
# print(full_contraction(lpdo_1_torch, lpdo_2_torch, pqc_torch))

Model created! Initial loss: -7.069338826638383e-05


In [124]:
lr = 0.01
optimizer = optim.Adam(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer,step_size=200, gamma=0.5)
num_steps = 1000
pbar = tqdm.tqdm(range(num_steps))
previous_loss = torch.inf
losses = []

print("Initial loss:", model.forward().item())

for step in pbar:
    optimizer.zero_grad()
    loss = model.forward()
    losses.append(loss.detach().numpy())
    loss.backward()
    optimizer.step()
    pbar.set_description(f"Loss={loss} - LR={lr}")
    if step > 100 and torch.abs(previous_loss - loss) < 1e-10:
        print("Early stopping loss difference is smaller than 1e-10")
        break
    previous_loss = loss.clone()
    
print(f'traning loss: {loss}')

print(model.forward().item())

Loss=-0.0016269913810192313 - LR=0.01:   0%|          | 1/1000 [00:00<02:10,  7.65it/s] 

Initial loss: -0.00017336520776156187


Loss=-0.45414781834631135 - LR=0.01: 100%|██████████| 1000/1000 [01:10<00:00, 14.09it/s]

traning loss: -0.45414781834631135
-0.4541565000398185


In [103]:
model.forward().item()
model.forward_debug().item()

-0.0011665454762127276

In [74]:
full_contraction(lpdo_1_torch, lpdo_2_torch, model.get_unitary_circuit())

tensor(-0.4341, dtype=torch.float64, grad_fn=<NegBackward0>)

In [102]:
full_contraction(lpdo_1_torch, lpdo_2_torch, pqc_torch)

tensor(-0.0012, dtype=torch.float64)

In [101]:
verify_aHerm(pqc_torch)

Checking unitarity of quantum circuit...

Tensor '0':
  Shape: (16, 16)
  Unitary: False
  Max error: 1.35e+00

Tensor '1':
  Shape: (16, 16)
  Unitary: False
  Max error: 1.43e+00

Tensor '2':
  Shape: (16, 16)
  Unitary: False
  Max error: 9.49e-01

Tensor '3':
  Shape: (16, 16)
  Unitary: False
  Max error: 1.14e+00

Tensor '4':
  Shape: (16, 16)
  Unitary: False
  Max error: 1.02e+00

Tensor '5':
  Shape: (16, 16)
  Unitary: False
  Max error: 1.08e+00

Tensor '6':
  Shape: (16, 16)
  Unitary: False
  Max error: 8.75e-01

Tensor '7':
  Shape: (16, 16)
  Unitary: False
  Max error: 1.12e+00

Tensor '8':
  Shape: (16, 16)
  Unitary: False
  Max error: 1.03e+00

Tensor '9':
  Shape: (16, 16)
  Unitary: False
  Max error: 1.06e+00

Tensor '10':
  Shape: (16, 16)
  Unitary: False
  Max error: 9.27e-01

Tensor '11':
  Shape: (16, 16)
  Unitary: False
  Max error: 9.36e-01

Tensor '12':
  Shape: (16, 16)
  Unitary: False
  Max error: 9.01e-01

Tensor '13':
  Shape: (16, 16)
  Unitary: Fal

False

#### Second time optimization

In [150]:
def update_lpdo(pqc, lpdo,n):

    pqc_cp = pqc.copy()
    for i in range (n):
        pqc_cp = pqc_cp.reindex({f'e{i}':f'f{i}'})
        pqc_cp = pqc_cp.reindex({f'a{i}':f'g{i}'})
        lpdo = lpdo.reindex({f'e{i}':f'f{i}'})
        lpdo = lpdo.reindex({f'a{i}':f'g{i}'})
        
    lpdo_new = (lpdo & pqc_cp)
    for i in range (n):
        lpdo_new = lpdo_new.reindex({f'ep{i}':f'e{i}'})
        lpdo_new = lpdo_new.reindex({f'ap{i}':f'a{i}'})

    return lpdo_new

In [151]:
depth = 4
framework = 'brickwall'
rand = True
val_iden = 0.01

pqc_new = get_pqc_torch(n, depth, framework, is_acl, rand=rand, val_iden=val_iden)
lpdo_1_new = update_lpdo(model.get_unitary_circuit(), lpdo_1_torch,n)

pqc_new.apply_to_arrays(lambda x: torch.tensor(x, dtype=torch.complex128))
lpdo_1_new.apply_to_arrays(lambda x: torch.tensor(x, dtype=torch.complex128))

C:\Users\Yuhan Liu\AppData\Local\Temp\ipykernel_7960\4245142164.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  lpdo_1_new.apply_to_arrays(lambda x: torch.tensor(x, dtype=torch.complex128))


In [152]:
model_new = TNModel(pqc_new, lpdo_1_new, lpdo_2_torch)
print("Model created! Initial loss:", model_new.forward().item())

  0%|          | 0/32 [00:00<?, ?it/s]

F=8.88 C=9.47 S=21.51 P=22.51: 100%|██████████| 32/32 [00:02<00:00, 11.42it/s]

Model created! Initial loss: -1.3473073849864347e-05


In [153]:
lr = 0.01
optimizer = optim.Adam(model_new.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer,step_size=200, gamma=0.5)
num_steps = 1000
pbar = tqdm.tqdm(range(num_steps))
previous_loss = torch.inf
losses = []

print("Initial loss:", model_new.forward().item())

for step in pbar:
    optimizer.zero_grad()
    loss = model_new.forward()
    losses.append(loss.detach().numpy())
    loss.backward()
    optimizer.step()
    pbar.set_description(f"Loss={loss} - LR={lr}")
    if step > 100 and torch.abs(previous_loss - loss) < 1e-10:
        print("Early stopping loss difference is smaller than 1e-10")
        break
    previous_loss = loss.clone()
    
print(f'traning loss: {loss}')

  0%|          | 0/1000 [00:00<?, ?it/s]

Initial loss: -1.3473073849864347e-05


Loss=-0.5087917135781531 - LR=0.01: 100%|██████████| 1000/1000 [05:37<00:00,  2.96it/s] 

traning loss: -0.5087917135781531


#### Misc

In [89]:
def circuits_equal(circuit1, circuit2, tolerance=1e-8):
    """Check if two circuits have the same parameter values."""
    tensors1 = list(circuit1.tensor_map.values())
    tensors2 = list(circuit2.tensor_map.values())
    
    if len(tensors1) != len(tensors2):
        return False
    
    for t1, t2 in zip(tensors1, tensors2):
        data1 = t1.data
        data2 = t2.data
        
        # Convert to same type if needed
        if isinstance(data1, torch.Tensor):
            data1 = data1.detach().cpu()
        if isinstance(data2, torch.Tensor):
            data2 = data2.detach().cpu()
        
        # Compare values
        if not torch.allclose(data1, data2, atol=tolerance):
            return False
    
    return True

# Usage
print("pqc_torch has same values as pqc_init?", 
      circuits_equal(pqc_torch, pqc_init))  # Likely True

print("pqc_torch has same values as optimized circuit?",
      circuits_equal(pqc_torch, model.get_unitary_circuit()))  # Likely False

pqc_torch has same values as pqc_init? False
pqc_torch has same values as optimized circuit? False


In [40]:
def verify_unitarity(pqc, tolerance=1e-6):

    print("Checking unitarity of quantum circuit...\n")
    
    all_unitary = True
    
    # Iterate through all tensors in the circuit
    for tag, tensor in pqc.tensor_map.items():
        # Get the tensor data
        data = tensor.data
        
        # Convert to numpy if it's a torch tensor
        if isinstance(data, torch.Tensor):
            data = data.detach().cpu().numpy()
        
        data = data.reshape(16, 16)

        n, m = data.shape
       
        # Check U† U = I
        U_dagger_U = np.dot(data.conj().T, data)
        identity = np.eye(n)
        is_unitary = np.allclose(U_dagger_U, identity, atol=tolerance)
        
        max_error = np.max(np.abs(U_dagger_U - identity))
        
        print(f"Tensor '{tag}':")
        print(f"  Shape: {data.shape}")
        print(f"  Unitary: {is_unitary}")
        print(f"  Max error: {max_error:.2e}\n")
        
        if not is_unitary:
            all_unitary = False

    return all_unitary


# Usage after training
final_circuit = model.get_unitary_circuit()
is_unitary = verify_unitarity(final_circuit)

if is_unitary:
    print("✓ All gates are unitary!")
else:
    print("✗ Some gates are NOT unitary!")

Checking unitarity of quantum circuit...

Tensor '0':
  Shape: (16, 16)
  Unitary: True
  Max error: 4.66e-15

Tensor '1':
  Shape: (16, 16)
  Unitary: True
  Max error: 5.11e-15

Tensor '2':
  Shape: (16, 16)
  Unitary: True
  Max error: 7.99e-15

Tensor '3':
  Shape: (16, 16)
  Unitary: True
  Max error: 5.77e-15

Tensor '4':
  Shape: (16, 16)
  Unitary: True
  Max error: 2.89e-15

Tensor '5':
  Shape: (16, 16)
  Unitary: True
  Max error: 5.33e-15

Tensor '6':
  Shape: (16, 16)
  Unitary: True
  Max error: 4.00e-15

Tensor '7':
  Shape: (16, 16)
  Unitary: True
  Max error: 5.33e-15

Tensor '8':
  Shape: (16, 16)
  Unitary: True
  Max error: 4.00e-15

Tensor '9':
  Shape: (16, 16)
  Unitary: True
  Max error: 2.89e-15

Tensor '10':
  Shape: (16, 16)
  Unitary: True
  Max error: 7.55e-15

Tensor '11':
  Shape: (16, 16)
  Unitary: True
  Max error: 5.33e-15

Tensor '12':
  Shape: (16, 16)
  Unitary: True
  Max error: 4.00e-15

Tensor '13':
  Shape: (16, 16)
  Unitary: True
  Max error

In [100]:
def verify_aHerm(pqc, tolerance=1e-6):

    print("Checking unitarity of quantum circuit...\n")
    
    all_unitary = True
    
    # Iterate through all tensors in the circuit
    for tag, tensor in pqc.tensor_map.items():
        # Get the tensor data
        data = tensor.data
        
        # Convert to numpy if it's a torch tensor
        if isinstance(data, torch.Tensor):
            data = data.detach().cpu().numpy()
        
        data = data.reshape(16, 16)

        n, m = data.shape
       
        # Check anti-Hermitian
        zero_matr = np.zeros((n,n))
        is_unitary = np.allclose(data - data.conj().T, zero_matr, atol=tolerance)
        
        max_error = np.max(np.abs(data - data.conj().T - zero_matr))
        
        print(f"Tensor '{tag}':")
        print(f"  Shape: {data.shape}")
        print(f"  Unitary: {is_unitary}")
        print(f"  Max error: {max_error:.2e}\n")
        
        if not is_unitary:
            all_unitary = False

    return all_unitary


# Usage after training
final_circuit = model.get_unitary_circuit()
is_unitary = verify_aHerm(final_circuit)

if is_unitary:
    print("✓ All gates are anti-Hermitian!")
else:
    print("✗ Some gates are anti-Hermitian!")

Checking unitarity of quantum circuit...

Tensor '0':
  Shape: (16, 16)
  Unitary: False
  Max error: 8.15e-01

Tensor '1':
  Shape: (16, 16)
  Unitary: False
  Max error: 7.62e-01

Tensor '2':
  Shape: (16, 16)
  Unitary: False
  Max error: 8.39e-01

Tensor '3':
  Shape: (16, 16)
  Unitary: False
  Max error: 8.42e-01

Tensor '4':
  Shape: (16, 16)
  Unitary: False
  Max error: 7.89e-01

Tensor '5':
  Shape: (16, 16)
  Unitary: False
  Max error: 8.14e-01

Tensor '6':
  Shape: (16, 16)
  Unitary: False
  Max error: 9.63e-01

Tensor '7':
  Shape: (16, 16)
  Unitary: False
  Max error: 9.69e-01

Tensor '8':
  Shape: (16, 16)
  Unitary: False
  Max error: 1.08e+00

Tensor '9':
  Shape: (16, 16)
  Unitary: False
  Max error: 9.25e-01

Tensor '10':
  Shape: (16, 16)
  Unitary: False
  Max error: 8.92e-01

Tensor '11':
  Shape: (16, 16)
  Unitary: False
  Max error: 6.58e-01

Tensor '12':
  Shape: (16, 16)
  Unitary: False
  Max error: 9.74e-01

Tensor '13':
  Shape: (16, 16)
  Unitary: Fal

In [1]:
from file_io import *
import numpy as np

In [85]:
# save_name = "pbcN16lr0.01num_steps4000sample50staircasedepth2icrm2icrm_bdy2"
# save_name = "h0p9N12lr0.01num_steps2000sample50staircasedepth4"
# save_name = "obccodeXN28lr0.01num_steps4000sample50staircasedepth4icrm2icrm_bdy2"
# save_name = "p03obcN14lr0.002num_steps10000sample50staircasedepth4"
save_name = "td_codeZobcN12lr0.002num_steps10000sample20staircasedepth4"
Dir = File_access()
test = Dir.get_back(save_name)
np.sort(test)

array([-0.68927246, -0.68442095, -0.68363956, -0.68105488, -0.67943049,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        ,  0.        ])